<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PytochCNNPractice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

import os
from google.colab import userdata

In [5]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('username and key activated')

username and key activated


In [6]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:31<00:00, 129MB/s]



In [7]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [8]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
pin_memory = (True if torch.cuda.is_available() else False)
print(pin_memory)

cpu
False


In [10]:
#transform
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

In [18]:
#dataset and dataload
train_data = datasets.ImageFolder(root=train_dir,transform=train_transform)
val_data = datasets.ImageFolder(root=validation_dir,transform=val_transform)

train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,num_workers=2,pin_memory=pin_memory)
val_loader = DataLoader(dataset=val_data,batch_size=32,num_workers=2,pin_memory=pin_memory)

In [25]:
# --- RUN THIS IN A SEPARATE CELL ---

# 1. Grab just ONE batch from the loader
test_images, test_labels = next(iter(train_loader))

# 2. Check the "Question" (Labels)
print(f"Labels from Loader: {test_labels.shape}")

# 3. Check the "Answer" (Model Output)
model.eval()
with torch.no_grad():
    # Pass the batch through the model
    test_outputs = model(test_images.to(device))

print(f"Model Predictions: {test_outputs.shape}")

# --- THE VERDICT ---
if test_outputs.shape == test_labels.shape:
    print("✅ PERFECT MATCH: No 'unsqueeze' or 'squeeze' needed for Loss!")
else:
    print("❌ MISMATCH: You need to fix the shapes in your loop.")


Labels from Loader: torch.Size([32])
Model Predictions: torch.Size([32, 1])
❌ MISMATCH: You need to fix the shapes in your loop.


In [12]:
#model
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(3,32,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(32,32,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(64,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.flatten = nn.Flatten()

    self.fc = nn.Sequential(
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Linear(128,1)
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)

    return x

In [13]:
model = CNN().to(device)

In [14]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [15]:
#training

epochs = 5

for epoch in range(epochs):
  model.train()

  train_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    output = model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()

    preds = (torch.sigmoid(output) > 0.5).int()
    train_correct += (preds.squeeze() == labels.squeeze()).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in val_loader:
      images = images.to(device)
      labels = labels.float().unsqueeze(1).to(device)

      optimizer.zero_grad()

      output = model(images)
      loss = loss_fn(output,labels)

      val_loss += loss.item()

      preds = (torch.sigmoid(output) > 0.5).int()
      val_correct += (preds.squeeze() == labels.squeeze()).sum().item()
      val_total += labels.size(0)

  val_accuracy = train_correct / train_total

  print(f"Epochs {epoch+1}/{epochs}")



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 